# Odds API data pipeline: offline demonstration

This walkthrough uses two synthetic matchups and fictional bookmakers. It calls the same modules as the CLI and dashboard.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from src.processing import clean_odds
from src.analysis import detect_discrepancies
raw = json.loads((root / "data/sample_odds.json").read_text())
cleaned = clean_odds(raw, market="h2h")
cleaned


,game_id,sport,commence_time,home_team,away_team,bookmaker,last_update,market,player_name,outcome,line,contract_line,price,decimal_odds,implied_prob,devig_prob
0,game_001,basketball_nba,2025-01-01T00:00:00Z,Los Angeles Lakers,Miami Heat,BookA,2025-01-01T00:00:00Z,h2h,None,Los Angeles Lakers,NaN,NaN,1.95,1.95,0.512821,0.530120
1,game_001,basketball_nba,2025-01-01T00:00:00Z,Los Angeles Lakers,Miami Heat,BookA,2025-01-01T00:00:00Z,h2h,None,Miami Heat,NaN,NaN,2.20,2.20,0.454545,0.469880
2,game_001,basketball_nba,2025-01-01T00:00:00Z,Los Angeles Lakers,Miami Heat,BookB,2025-01-01T00:00:00Z,h2h,None,Los Angeles Lakers,NaN,NaN,1.85,1.85,0.540541,0.519481
3,game_001,basketball_nba,2025-01-01T00:00:00Z,Los Angeles Lakers,Miami Heat,BookB,2025-01-01T00:00:00Z,h2h,None,Miami Heat,NaN,NaN,2.00,2.00,0.500000,0.480519
4,game_002,basketball_nba,2025-01-02T01:00:00Z,Denver Nuggets,Phoenix Suns,BookA,2025-01-02T00:00:00Z,h2h,None,Denver Nuggets,NaN,NaN,1.70,1.70,0.588235,0.552632
5,game_002,basketball_nba,2025-01-02T01:00:00Z,Denver Nuggets,Phoenix Suns,BookA,2025-01-02T00:00:00Z,h2h,None,Phoenix Suns,NaN,NaN,2.10,2.10,0.476190,0.447368


## Normalize within each bookmaker

Each complete two-way market sums to one after proportional margin removal. Different bookmakers are separate groups.

In [2]:
cleaned.groupby(["game_id", "bookmaker"])["devig_prob"].sum()

game_id   bookmaker
game_001  BookA        1.0
          BookB        1.0
game_002  BookA        1.0
Name: devig_prob, dtype: float64

In [3]:
report = detect_discrepancies(cleaned, market_key="h2h")
report

,game_id,market,player_name,contract_line,home_team,away_team,outcome,line,best_bookmaker,best_price,implied_prob,arbitrage_margin
0,game_001,h2h,None,NaN,Los Angeles Lakers,Miami Heat,Miami Heat,NaN,BookA,2.20,0.454545,3.3735
1,game_001,h2h,None,NaN,Los Angeles Lakers,Miami Heat,Los Angeles Lakers,NaN,BookA,1.95,0.512821,3.3735
2,game_002,h2h,None,NaN,Denver Nuggets,Phoenix Suns,Phoenix Suns,NaN,BookA,2.10,0.476190,NaN
3,game_002,h2h,None,NaN,Denver Nuggets,Phoenix Suns,Denver Nuggets,NaN,BookA,1.70,0.588235,NaN


## Export a reproducible report

The arbitrage column is a theoretical return on total stake. Sample prices do not describe current offers.

In [4]:
(root / "artifacts").mkdir(exist_ok=True)
report.to_csv(root / "artifacts/demo_best_odds_report.csv", index=False)